# Dictionaries

**Topic:** `3.4` &nbsp;|&nbsp; **Module 3: Core Data Structures**

*Python's hash map: unique hashable keys to values in amortised O(1), insertion-ordered iteration, and the three silent failures - get() masking absence, shallow copies sharing state, mutation during iteration.*

> *Course:* Python Master Course &nbsp;|&nbsp; **Project:** ROBO-X (M3)

### Navigate

* [Lesson](lesson.ipynb) &nbsp;|&nbsp; [Exercises](exercises.ipynb) &nbsp;|&nbsp; [Solutions](solution.ipynb) &nbsp;|&nbsp; [Research](research.ipynb) &nbsp;|&nbsp; [Quiz](quiz.ipynb) &nbsp;|&nbsp; [Mini-project](mini_project.ipynb) &nbsp;|&nbsp; [Robotics challenge](robotics_challenge.ipynb)

Module index: [03_data_structures/README.md](../README.md)

## 1. Why This Topic Matters

Config, counters, indices, JSON payloads and parameter servers are all dicts, and the bugs are silent ones: a get() typo produces plausible garbage, a shallow copy corrupts live settings through a shared inner object, and a half-applied update is visible to every reader. The dict is where reference semantics meets production data.

## 2. Learning Objectives

By the end of this topic you will be able to:

1. Choose [] versus .get based on whether absence is an error.
2. Count, group and index with dict idioms in linear time.
3. Explain why keys must be hashable and how lookup stays O(1).
4. Diagnose and repair shallow-copy aliasing in nested records.
5. Iterate safely by snapshotting or rebuilding instead of mutating.
6. Publish updated mappings atomically by building then rebinding.

## 3. Prerequisites

This topic assumes you already have:

- Topic 3.3 Sets

## 4. Mental Model

**Mental model: a dict is a wall of numbered pigeonholes with the numbers written on the items, not on the holes.** The hash of a key says which section of the wall to visit; the key is checked at the slot. Retrieval is a direct walk, not a search - but an unknown item has no hole at all, which is why `d[k]` raises rather than guesses.

| Question | Answer | Consequence |
| --- | --- | --- |
| Can two keys share a slot? | no - keys are unique | re-assignment replaces, never duplicates |
| What if the key is missing? | `[]` raises KeyError | get(k, default) is the explicit opt-out |
| Is a copy independent? | outer yes, values shared | deepcopy or rebuild nested levels |
| Iteration order? | insertion order (3.7+) | reproducible output without sorting |
| Mutable keys? | impossible - they would not hash | lists/dicts as keys raise TypeError |
| Mutating while iterating? | RuntimeError | snapshot with list(d) first |

## 5. Terminology

Vocabulary used consistently throughout the course.

| Term | Meaning |
| --- | --- |
| Dictionary | A mutable mapping from unique hashable keys to values. |
| Key | A hashable object identifying one entry; unique per dict. |
| Value | Any object bound to a key; may itself be mutable. |
| KeyError | Raised by d[k] when the key is absent - fail-fast access. |
| get(k, default) | Absent-safe access returning an explicit default. |
| View | A live, read-only window over keys, values or items. |
| Insertion order | The guaranteed iteration order of a dict since 3.7. |
| Shallow copy | A new dict whose values are references to the originals. |
| Hash table | The array-of-slots structure giving amortised O(1) lookup. |
| Mapping | The protocol of key-based access a dict implements. |

## 6. Conceptual Explanation

A dict is Python's hash map: a set of unique, hashable **keys**, each bound to a **value**, retrievable in amortised O(1) time. It is the language's default record type - a config blob, a status snapshot, a JSON object - because it states the association directly: `config['timeout'] = 30` reads as 'timeout means 30' in a way two parallel lists never could.

Three properties define the type. **Keys are unique**: assigning an existing key replaces its value, it does not create a second entry - which makes dicts natural for grouping and counting. **Keys must be hashable**: strings, numbers and tuples qualify; lists and dicts do not. **Order is insertion order**: guaranteed since Python 3.7, so a dict iterates exactly as data was added - a contract you can rely on for reproducible output.

In [ ]:
config = {'timeout': 30, 'retries': 3}
print('value:', config['timeout'])

config['timeout'] = 45            # update in place
config['timeout'] = 50            # unique key: REPLACES, no duplicate
print('keys :', list(config))      # insertion order preserved

try:
    print(config['missing'])
except KeyError as exc:
    print('missing key raises:', exc)

The KeyError is the dict's most important behaviour: `d[k]` is a **contract** - the key must exist, or the program fails loudly at the point of use. When absence is normal, you opt into a different operation deliberately: `d.get(k, default)` for a fallback, `k in d` to test first, `d.setdefault(k, factory)` to fill on demand. Each answers a different question, and choosing `get` when you meant `[]` is how missing configuration becomes a silent wrong number.

Under the hood the dict is a compact hash table with indices derived from key hashes - which is why keys must be immutable (a changing hash would lose the entry) and why lookup stays O(1) regardless of size. Values have no such restriction: anything at all may live on the value side, including other dicts and lists.

## 7. Formal Theory

Formally a dict implements a finite mapping from hashable keys to values. Construction, indexing, deletion and membership are amortised O(1) expected; iteration walks the insertion-ordered key sequence. The identities worth memorising:

```text
d[k] raises KeyError if k is absent   |   d.get(k, default) returns default   |   k in d tests without fetching
```

In [ ]:
levels = {'info': 20, 'warn': 10, 'fault': 1}

print('direct :', levels['warn'])
print('get    :', levels.get('trace', 0))
print('test   :', 'fault' in levels)
print('pop    :', levels.pop('info'))
print('now    :', dict(levels))

| Operation | Time | Notes |
| --- | --- | --- |
| d[k], d[k] = v | amortised O(1) | hash + probe |
| d.get(k, default) | amortised O(1) | absence returns the default |
| k in d | amortised O(1) | tests presence without fetching |
| del d[k] | amortised O(1) | KeyError if absent |
| d.update(other) | O(len(other)) | merge, overwriting on conflict |
| dict(d) | O(n) | shallow copy - values are shared |
| iteration / keys() | O(n) | in insertion order |

The views - `d.keys()`, `d.values()`, `d.items()` - are live, read-only views over the dict, not snapshots: iterating them while the dict changes raises RuntimeError exactly as sets do, and `list(d) - d.keys()` style set operations work because keys() behaves like a set. Values may be unhashable - `[d[v] for v in d.values()]` is perfectly legal - because only the key side must hash.

Two construction forms are worth distinguishing: `dict(pairs)` builds from (key, value) pairs or from a mapping, while the literal `{k: v}` and the comprehension `{k_expr: v_expr for ...}` build directly. A comprehension that can be expressed as `dict(zip(keys, values))` is usually clearer - and zip's length rule (shortest wins) becomes a correctness question you must answer deliberately.

## 8. Syntax

Literal creation, safe access, merging, and the comprehension form.

In [ ]:
telemetry = {'rate_hz': 100, 'channel': 'imu'}   # literal
empty = {}                                       # a real dict
from_pairs = dict([('a', 1), ('b', 2)])          # from pairs
from_keys = dict.fromkeys(('x', 'y'), 0)         # shared default

print('literal:', telemetry)
print('pairs  :', from_pairs)
print('defaults:', from_keys)

rate = telemetry.get('rate_hz', 0)
absent = telemetry.get('gain', 1.0)
print('rate:', rate, ' absent default:', absent)

The anti-pattern - get() with a default where a missing key is a real error:

```text
timeout = config.get('timeout')      # None when missing...
socket.settimeout(timeout)           # ...and fails HERE, far away

timeout = config['timeout']          # fails at the source: KeyError
# names the exact missing key, at the line that assumed it exists
```

> **get() hides provenance**
>
> `d.get(k, 0)` turns a typo'd key into a plausible zero that flows downstream. Use `[]` when the key must exist, and `get` only when absence is a documented, expected case with a meaningful default.

Merging and the mutating update, plus the comprehension that reads like its SQL:

In [ ]:
base = {'host': 'localhost', 'port': 8080}
override = {'port': 9090, 'debug': True}

merged = base | override                 # new dict, override wins
print('merged :', merged)

combined = {k: v for k, v in merged.items() if v is not None}
print('filtered:', combined)

counts = {}
for tag in ('a', 'b', 'a'):
    counts[tag] = counts.get(tag, 0) + 1
print('counts :', counts)

## 9. Basic Examples

**Example 1 - build, fetch, and update a status record.**

In [ ]:
status = {}
status['name'] = 'rover-01'
status['battery_pct'] = 88.5
status.update({'mode': 'explore'})

print('status :', status)
print('name   :', status['name'])
print('keys   :', list(status))
print('values :', list(status.values()))

**Example 2 - iterate items in insertion order.**

In [ ]:
limits = {'temp_max': 60.0, 'temp_min': -10.0, 'rate_max': 250}
for name, limit in limits.items():
    print(f'{name:<10} {limit}')

print('all names sorted:', sorted(limits))

**Example 3 - counting with get(), the pattern behind Counter.**

In [ ]:
words = 'go hold go stop go'.split()
counts = {}
for word in words:
    counts[word] = counts.get(word, 0) + 1
print('counts:', counts)
print('most frequent:', max(counts, key=counts.get))

## 10. Intermediate Examples

**Grouping with setdefault** - the one-liner that builds dict-of-lists: the key's list is created on first sight, appended to on every sight after.

In [ ]:
by_kind = {}
events = [('imu', 'boot'), ('gps', 'fix'), ('imu', 'drift')]

for kind, detail in events:
    by_kind.setdefault(kind, []).append(detail)

print('grouped:', by_kind)

**Reversing a mapping** exposes the uniqueness rule: two keys with the same value collapse under inversion, so the design must say which one wins.

In [ ]:
ports = {'http': 80, 'https': 443, 'alt_https': 443}
reversed_map = {v: k for k, v in ports.items()}
print('reversed:', reversed_map)   # 443 appears once - last key wins

ambiguous = [v for v in ports.values() if list(ports.values()).count(v) > 1]
print('collision values:', sorted(set(ambiguous)))

**Nested records and the copy trap preview.** Reading through two levels is ordinary; copying the outer dict and editing the nested record is where aliasing returns.

In [ ]:
robots = {
    'rover-01': {'x': 0.0, 'y': 0.0},
    'rover-02': {'x': 3.0, 'y': 1.0},
}

snapshot = dict(robots)             # SHALLOW copy
snapshot['rover-01']['x'] = 99.0    # mutates the SHARED inner dict
print('original x :', robots['rover-01']['x'])   # 99.0 - corrupted!
print('snapshot x :', snapshot['rover-01']['x'])

## 11. Advanced Examples

**Mutation during iteration** raises RuntimeError the moment a key is added or deleted - the guard exists because a resizing table invalidates the iterator. The clean patterns: iterate a snapshot of the keys, or rebuild into a new dict.

In [ ]:
thresholds = {'a': 1, 'b': 2, 'c': 3}
try:
    for key in thresholds:
        if thresholds[key] < 2:
            del thresholds[key]
except RuntimeError as exc:
    print('caught:', exc)

thresholds = {'a': 1, 'b': 2, 'c': 3}
kept = {k: v for k, v in thresholds.items() if v >= 2}
print('rebuilt :', kept)

**setdefault's dark side.** It eagerly builds the default even when the key exists - so `d.setdefault(k, [])` inside a loop over a million keys allocates a million empty lists that are immediately discarded. The `get`-then-branch form or a `defaultdict` avoids it.

In [ ]:
import time

keys = [f'k{i}' for i in range(200000)]

start = time.perf_counter()
d1 = {}
for k in keys:
    d1.setdefault(k, [])       # builds a throwaway list every hit
setdefault_s = time.perf_counter() - start

start = time.perf_counter()
d2 = {}
for k in keys:
    if k not in d2:
        d2[k] = []             # allocate only on a real miss
branch_s = time.perf_counter() - start

print(f'setdefault {setdefault_s:.3f}s vs branch {branch_s:.3f}s')

> **dict.fromkeys shares one default**
>
> `dict.fromkeys(names, 0)` is fine for immutable defaults - but `dict.fromkeys(names, [])` gives every key the SAME list. Build with a comprehension instead whenever the default is mutable.

In [ ]:
bad = dict.fromkeys(('a', 'b'), [])
bad['a'].append(1)
print('shared!', bad)                 # {'a': [1], 'b': [1]}

good = {k: [] for k in ('a', 'b')}
good['a'].append(1)
print('independent:', good)

## 12. Code Walkthrough

**A telemetry aggregator that cannot silently drop data.** It counts per-channel samples, tracks a running sum for means, rejects unknown channels loudly, and never mutates while iterating.

In [ ]:
def summarise(samples, known_channels):
    """Aggregate (channel, value) pairs into per-channel stats."""
    counts = dict.fromkeys(known_channels, 0)
    totals = dict.fromkeys(known_channels, 0.0)
    for channel, value in samples:
        if channel not in counts:
            raise KeyError(f'unknown channel {channel!r}')
        counts[channel] += 1
        totals[channel] += value
    means = {
        ch: round(totals[ch] / counts[ch], 3)
        for ch in counts
        if counts[ch]           # zero-division guard per channel
    }
    return {'counts': counts, 'means': means}

out = summarise([('imu', 1.0), ('imu', 3.0), ('gps', 2.0)], ['imu', 'gps'])
print('counts:', out['counts'])
print('means :', out['means'])

- `dict.fromkeys(channels, 0)` seeds both tables from the known channel list - the counts table doubles as the whitelist.
- The `channel not in counts` guard turns an unknown channel into a loud KeyError instead of a quietly created entry - the get() trap avoided by design.
- Totals stay separate from counts so the mean divides by real observations, not by channel count.
- The means comprehension filters `if counts[ch]`, handling channels that received zero samples without a ZeroDivisionError.

## 13. Line-by-Line Explanation

1. `counts = dict.fromkeys(known_channels, 0)` - zero counts for every legal channel; immutable default 0, so sharing is safe.
2. `for channel, value in samples:` - unpacks each pair directly; no indexing into tuples.
3. `if channel not in counts: raise KeyError(...)` - membership first, so the error names the channel and no phantom key is created.
4. `counts[channel] += 1` - fetch-add-store on an existing key; amortised O(1).
5. `totals[channel] += value` - a parallel table, so means need no second pass over samples.
6. `{ch: round(...) for ch in counts if counts[ch]}` - the filter guards division; channels with no samples simply do not appear in the means.
7. `return {'counts': counts, 'means': means}` - a record of records - dicts nesting naturally is what makes the return type self-describing.

## 14. Common Mistakes

**Mistake 1 - get() where the key must exist.**

```text
cfg = {'port': 8080}          # 'host' forgotten
host = cfg.get('host')        # None - the bug escapes
url = f'http://{host}:{cfg.get("port")}'
print(url)                    # 'http://None:8080' - plausible garbage

host = cfg['host']            # KeyError AT THE SOURCE - fixed fast
```

**Mistake 2 - assuming a copy is independent at every level.**

```text
original = {'cfg': {'mode': 'a'}}
clone = original.copy()          # shallow!
clone['cfg']['mode'] = 'b'
print(original['cfg']['mode'])   # 'b' - the inner dict is shared
```

**Mistake 3 - mutating the dict while iterating it.**

```text
d = {'a': 1, 'b': 2}
for k in d:
    if d[k] == 1:
        d['c'] = 3        # RuntimeError: dictionary changed size

for k in list(d):         # snapshot the keys first
    ...
```

**Mistake 4 - keys that cannot hash.**

```text
d = {}
d[['x', 'y']] = 1         # TypeError: unhashable type: 'list'
d[('x', 'y')] = 1         # tuple key - the fix
```

**Mistake 5 - iterating .values() while assigning keys.**

```text
for v in d.values():
    d['derived'] = v * 2   # RuntimeError - the view is live

for v in list(d.values()):  # snapshot the view
    ...
```

## 15. Debugging Techniques

When a dict 'loses' an entry, print the keys BEFORE the suspect write - a KeyError with the exact name, plus a sorted key dump, identifies typos and case mismatches in one step.

In [ ]:
config = {'Timeout': 30}          # capital T - the classic typo
print('keys:', sorted(config))
try:
    config['timeout']
except KeyError as exc:
    print('missing:', exc)
    folded = {k.lower(): k for k in config}
    print('case-insensitive hit?', folded.get('timeout'))

When a copy changed the original, print identity of the nested values - `id()` proves whether two paths reach one object.

In [ ]:
original = {'cfg': {'mode': 'a'}}
clone = dict(original)

print('outer distinct?', original is not clone)                 # True
print('inner shared  ?', original['cfg'] is clone['cfg'])   # True
print('fix: copy.deepcopy or rebuild inner dicts')

> **views are live, not snapshots**
>
> `list(d.items())` snapshots; `d.items()` does not. If a RuntimeError mentions changed size mid-loop, the view is the culprit - wrap the iteration target in list().

## 16. Best Practices

- Use `d[k]` for required keys - let KeyError fail fast at the source; reserve `get(k, default)` for documented optional keys.
- Never mutate a dict while iterating it; snapshot with list() or rebuild with a comprehension.
- Copy nested data with copy.deepcopy (or rebuild inner dicts explicitly) - dict() and .copy() are shallow.
- Prefer `d | other` or update-into-a-new-dict over mutating a shared mapping.
- Keys: strings for config, tuples for coordinates - keep them hashable and stable.
- Values: anything goes, but document when a value is itself mutable state.
- Iterate .items() when both key and value are needed - it is one lookup, not two.
- Use dict.fromkeys only with immutable defaults; comprehension for mutable ones.
- Sort at the boundary (sorted(d)) when output order must be reproducible.

## 17. Performance Considerations

The dict is the reason Python beats linear-scan languages at key-value work: amortised O(1) lookup from a hash probe, regardless of size. Everything expensive in dict-land is either O(n) by nature (iteration, copying, merging many) or a misuse (building keys by string concatenation, scanning values for a key).

| Pattern | Cost | Preferred alternative |
| --- | --- | --- |
| `k in d.values()` to find a key | O(n) per call | invert the mapping once, then O(1) |
| `d.get(k, 0) + 1` counting | O(1) - correct | collections.Counter for richer stats |
| Rebuilding d in a loop | O(n) per rebuild | update in place or accumulate elsewhere |
| `dict.copy()` for nested state | shallow O(n) | copy.deepcopy when inner state is shared |
| `d[k] = d.get(k, 0) + 1` | O(1) amortised | the right pattern - keep it |
| Searching every value for x | O(n) | build a reverse index: {v: k for k, v in d.items()} |

In [ ]:
import time

n = 100_000
mapping = {f'key{i}': i for i in range(n)}
keys = list(mapping)

start = time.perf_counter()
hits = sum(1 for k in keys if k in mapping)
dict_s = time.perf_counter() - start

pairs = list(mapping.items())
start = time.perf_counter()
hits2 = sum(1 for k in keys if any(pk == k for pk, _ in pairs))
scan_s = time.perf_counter() - start

print(f'dict lookup {dict_s:.4f}s vs linear scan {scan_s:.4f}s')
print(f'dict is {scan_s / max(dict_s, 1e-9):.0f}x faster here')

Two structural notes. Insertion is amortised O(1) but *deletion* leaves the table's slot bookkeeping to be maintained lazily - still amortised constant. And iteration cost is O(n) whether you walk keys, values or items, because all three views traverse the same table; `list(d.items())` doubles the peak memory for the snapshot, which matters at a million entries.

## 18. Pythonic Approaches

Idiomatic dict code exploits three affordances: items() iteration, comprehensions that build the whole mapping in one expression, and the merge operator - each replacing an explicit loop the reader would otherwise have to audit.

In [ ]:
names = ('ada', 'grace', 'alan')
scores = (91, 88, 95)

# not pythonic
board = {}
for i in range(len(names)):
    board[names[i]] = scores[i]

# pythonic - zip pairs them, the comprehension names the result
board = {name: score for name, score in zip(names, scores)}
print('board:', board)
print('above 90:', {n: s for n, s in board.items() if s > 90})

- Iterate `.items()` when you need both sides - it is one traversal, not two lookups.
- `dict(zip(keys, values))` beats an index loop for parallel sequences.
- `{**base, **overrides}` or `base | overrides` merges without mutating either operand.
- Invert deliberately: `{v: k for k, v in d.items()}` - and document which key wins collisions.
- `d.keys() & other.keys()` answers set questions about mappings directly.
- Use `sorted(d, key=d.get)` to order by value in one readable expression.
- defaultdict(list) or Counter when grouping/counting - but know what mutates on access.

## 19. Robotics Connection

A rover's parameter server is a dict: hundreds of tunables keyed by name, read on every control tick, updated between runs. The safety requirements map directly onto dict semantics - missing keys must fail loudly, updates must not mutate a config another component is reading, and the hot read path must stay O(1).

In [ ]:
import sys
from pathlib import Path

here = Path.cwd()
root = next((p for p in [here, *here.parents] if (p / 'shared').is_dir()), None)
if root is not None:
    sys.path.insert(0, str(root))

from shared.robo_x_sim import SimulatedRobot  # noqa: E402

robot = SimulatedRobot(name='rover-01', battery_wh=48.0)
params = {'speed_mps': 0.4, 'gains': {'kp': 1.2, 'ki': 0.05}}

snapshot = {'speed_mps': params['speed_mps'],
            'gains': dict(params['gains'])}      # isolate the nest
snapshot['gains']['kp'] = 2.0                    # tuning this run

print('live kp :', params['gains']['kp'])         # unchanged
print('snap kp :', snapshot['gains']['kp'])
print('battery :', round(robot.status()['battery_pct'], 1), '%')

The boundary-copy habit is the engineering rule: a component that reads parameters takes a snapshot with nested copies; a component that publishes new parameters returns a fresh dict rather than editing the shared one. Pair it with `params['required_key']` for mandatory tunables so a typo fails at load time, not at the moment the rover depends on the value.

## 20. Engineering Example

A rate limiter keyed by client id is classic dict engineering: each client's window lives in its own nested record, absence is normal (first request), and the design must not leak state between clients or between windows.

In [ ]:
import time

def make_limiter(max_per_window=5, window_s=60.0):
    """Return an allow(client_id) callable - dict-backed window counter."""
    windows = {}

    def allow(client_id):
        now = time.monotonic()
        window = windows.get(client_id)
        if window is None or now - window['reset'] > window_s:
            windows[client_id] = {'count': 1, 'reset': now}
            return True
        if window['count'] >= max_per_window:
            return False
        window['count'] += 1          # nested dict mutated in place
        return True

    return allow

allow = make_limiter(max_per_window=2, window_s=60.0)
print('first  :', allow('alpha'))
print('second :', allow('alpha'))
print('third  :', allow('alpha'))   # over the limit
print('other  :', allow('beta'))    # independent record

Note the shape: a closure over one dict, `.get` for the genuinely optional first sighting, `[]` once the record exists, and a nested dict mutated in place for the counter. Each client's record is independent precisely because the inner dicts are built fresh on first use - not shared via fromkeys.

## 21. Guided Practice

1. Build a dict three ways - literal, dict(pairs) and fromkeys - and print each with its type.
2. Fetch a missing key with `[]` (read the KeyError) and with .get(k, default); explain when each is correct.
3. Count word frequencies with .get(), then find the most frequent word with max(..., key=counts.get).
4. Reproduce the shallow-copy corruption on a nested record, then fix it with a dict rebuild of the inner level.
5. Trigger the RuntimeError by adding a key during iteration, then iterate list(d) and confirm it succeeds.

In [ ]:
pairs = [('rate', 100), ('channel', 'imu')]
print('literal:', {'rate': 100})
print('pairs  :', dict(pairs))
print('fromkeys:', dict.fromkeys(('a', 'b'), 0))

cfg = {'port': 8080}
try:
    cfg['host']
except KeyError as exc:
    print('[] raises:', exc)
print('get default:', cfg.get('host', 'localhost'))

words = 'go hold go stop go'.split()
counts = {}
for w in words:
    counts[w] = counts.get(w, 0) + 1
print('counts:', counts, ' top:', max(counts, key=counts.get))

## 22. Knowledge Checks

Answer these before moving on. Full answers are in [solution.ipynb](solution.ipynb).

- **Q1.** What does `cfg.get('timeout')` return when 'timeout' is absent?
- **Q1.** Which key type is legal in a dict?
- **Q1.** A rover parameter server must update its config without disturbing readers mid-tick. Which design is safest?
- **Q1.** What is the cost of finding a VALUE's key by scanning a dict?

## 23. Summary

A dict maps unique hashable keys to values in amortised O(1) time, iterates in insertion order, and replaces - never duplicates - a key that is assigned twice. It is Python's record type: config, status, counters, indices - anywhere data is a set of named fields rather than a sequence.

The three failure modes structure the topic. **Silent absence**: `d[k]` raises loudly at the source, while `.get(k, default)` can turn a typo into plausible garbage downstream - reserve it for documented optional keys. **Shared nesting**: every built-in copy is shallow, so editing a copied record mutates the original's inner dict; deepcopy or rebuild inner levels at boundaries. **Mutation during iteration**: views are live and raise RuntimeError - snapshot with list(d) or rebuild with a comprehension.

The performance story is why dicts dominate: constant-time membership and fetch mean an inverted mapping replaces every value scan, counting is one get-and-add per item, and grouping is a setdefault/comprehension one-liner. Practise the discipline - `[]` for required keys, get for optional, sorted output at boundaries - and the record type stops being a source of bugs.

## 24. Key Takeaways

- Dicts map unique hashable keys to values; assigning an existing key replaces it.
- `d[k]` raises KeyError on absence - use it for required keys; .get(k, default) only for documented optional ones.
- Iteration follows insertion order (guaranteed since 3.7); sort when output must be canonical.
- Every built-in copy is shallow - nested dicts stay shared; deepcopy or rebuild at boundaries.
- Never mutate a dict while iterating it - snapshot with list(d) or rebuild with a comprehension.
- Keys must be hashable (str, int, tuple); values can be anything.
- Invert a mapping ({v: k for ...}) instead of scanning values, and document collision policy.
- dict.fromkeys shares one default object - use a comprehension for mutable defaults.

## 25. Further Exploration

- Read about dict's compact insertion-order layout (PEP 468 and the How Dicts Work section of the docs).
- Time 100000 lookups against a dict versus a list of pairs, and plot both against n.
- Explore collections.defaultdict, Counter and OrderedDict - and write one paragraph on when each earns its place.
- Investigate `d | other` versus update(): which mutates, and which operands survive?
- Study json.dumps(..., sort_keys=True) as the reproducible serialisation of a dict.